# Where and what to buy

**Brief: Where (neighbourhoods) + What (property types and mix).**

- Revenue, nightly price and occupancy by neighbourhood and type
- Which segments have the best yield once purchase price is included
- Watch out: segments with few listings (`n`) have unreliable medians

In [7]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
T = config.TABLES   # run `python run.py` from the repo root first if outputs/ is empty

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

## Data
- `seg`: neighbourhood × type × bedrooms (active entire homes)
- `uw`: segments + purchase price, costs, NOI, net yield

### From 01 (keep in mind)
- **Occupancy & revenue** = Inside Airbnb's estimates (`estimated_occupancy_l365d`, `estimated_revenue_l365d`), 12 months to 15 June 2026. Estimates, not real bookings.
- **Price** = one night's quote around June–July 2026 (World Cup period), so revenue may be a bit high.
- **`is_active`** = reviewed in the last 12 months. Only active listings are used for segments.
- Cleaned data: 6,509 → 4,401 listings (no price, price outside $30–$2,000, 30+ night minimum removed).

In [8]:
seg = pd.read_csv(T / "segments.csv", dtype={"bedrooms_grp": str})
uw = pd.read_csv(T / "segments_underwritten.csv", dtype={"bedrooms_grp": str})

## Quick look

In [9]:
uw[uw["n"] >= 5].sort_values("net_yield", ascending=False).head(20)

,neighbourhood_cleansed,bucket,bedrooms_grp,n,adr_median,occ_median,rev_median,rev_p25,rev_p75,rating,share_commercial,share_licensed_est,side,gvr_type,benchmark_price_cad,price_cad,noi,cash_in,net_yield
97,Strathcona,condo,1,10,406.50,0.46,"83,406.00","30,499.75","89,265.75",4.96,0.00,1.00,East,Apartment,635100,"539,835.00","46,430.63","586,428.40",0.08
14,Downtown Eastside,condo,3+,13,491.00,0.33,"81,345.00","20,790.00","117,612.00",4.67,0.46,0.85,East,Apartment,635100,"952,650.00","43,770.10","1,015,756.00",0.04
83,Riley Park,condo,2,7,353.20,0.31,"65,695.00","26,764.00","77,775.00",4.88,0.00,1.00,East,Apartment,635100,"730,365.00","33,638.45","784,579.60",0.04
12,Downtown Eastside,condo,1,152,368.16,0.35,"49,140.00","17,515.50","75,666.75",4.88,0.14,0.95,East,Apartment,635100,"539,835.00","22,787.10","586,428.40",0.04
6,Downtown,condo,1,364,412.45,0.35,"53,150.00","21,288.25","83,669.25",4.88,0.13,0.95,West,Apartment,751400,"638,690.00","25,257.43","689,237.60",0.04
7,Downtown,condo,2,283,551.57,0.33,"64,008.00","24,764.00","102,789.00",4.88,0.17,0.98,West,Apartment,751400,"864,110.00","32,073.19","923,674.40",0.03
40,Kensington-Cedar Cottage,condo,1,7,275.50,0.44,"44,095.00","40,791.00","49,275.00",4.82,0.14,1.00,East,Apartment,635100,"539,835.00","19,306.04","586,428.40",0.03
8,Downtown,condo,3+,121,575.00,0.33,"75,345.00","28,758.00","125,715.00",4.82,0.38,0.93,West,Apartment,751400,"1,127,100.00","39,106.75","1,197,184.00",0.03
54,Kitsilano,condo,1,33,370.88,0.31,"46,221.00","14,858.00","68,544.00",4.88,0.12,1.00,West,Apartment,751400,"638,690.00","20,476.42","689,237.60",0.03
111,West End,condo,2,24,535.78,0.28,"56,709.50","26,014.50","98,831.25",4.91,0.17,0.96,West,Apartment,751400,"864,110.00","27,037.22","923,674.40",0.03


In [10]:
uw.groupby("bucket")[["rev_median", "price_cad", "net_yield"]].median()

,rev_median,price_cad,net_yield
bucket,,,
condo,"31,797.00","730,365.00",0.01
house,"35,710.50","2,899,700.00",0.01


## Analysis
_Start here._

### Step 1: Demand by neighbourhood
Active entire-home listings only (what DDD would run). Typical (median) value per neighbourhood. Hover a bar to see the typical nightly price.

In [11]:
import plotly.express as px

df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)
d = df[df["is_active"] & (df["room_type"] == "Entire home/apt")]

hood = (d.groupby("neighbourhood_cleansed")
          .agg(listings=("id", "size"), revenue=("revenue", "median"),
               occupancy=("occupancy", "median"), price=("price_num", "median"))
          .reset_index())
hood

,neighbourhood_cleansed,listings,revenue,occupancy,price
0,Arbutus Ridge,64,"38,682.00",0.30,400.00
1,Downtown,779,"59,328.00",0.35,466.00
2,Downtown Eastside,290,"44,043.00",0.28,401.18
3,Dunbar Southlands,125,"32,886.00",0.18,385.72
4,Fairview,18,"45,239.00",0.26,435.75
5,Grandview-Woodland,128,"36,199.50",0.25,350.88
6,Hastings-Sunrise,176,"42,266.50",0.34,312.88
7,Kensington-Cedar Cottage,217,"40,741.00",0.33,350.00
8,Kerrisdale,46,"32,396.50",0.20,399.98
9,Killarney,73,"24,960.00",0.21,319.53


In [12]:
def bar(col, title, fmt, note=""):
    h = hood.sort_values(col)
    fig = px.bar(h, x=col, y="neighbourhood_cleansed", orientation="h", title=title,
                 custom_data=["price", "listings"], color_discrete_sequence=["#2a78d6"])
    fig.update_traces(hovertemplate="<b>%{y}</b><br>" + title.split(" (")[0] + ": %{x:" + fmt + "}"
                      "<br>Typical nightly price: $%{customdata[0]:,.0f}<br>Listings: %{customdata[1]}<extra></extra>")
    if note:
        fig.add_annotation(text=note, xref="paper", yref="paper", x=0, y=1.06, showarrow=False,
                           xanchor="left", font=dict(size=12, color="#52514e"))
    fig.update_layout(height=600, yaxis_title=None, xaxis_title=None, plot_bgcolor="white")
    fig.update_xaxes(showgrid=True, gridcolor="#eee", tickformat=fmt)
    fig.show()

bar("revenue", "Typical yearly revenue per listing (CAD)", "$,.0f",
    note="Yearly revenue = occupancy (nights booked) × nightly price · 12 months to June 2026")
bar("occupancy", "Typical occupancy (share of year booked)", ".0%")
bar("listings", "Number of active whole-home listings", ",d",
    note="Supply: more listings = proven demand, but also more competition")